# M4 matching spike — NKC → Goodreads (English, 2003+, n=500)

> **Note:** `data/interim/matching_spike_results.csv` written by this notebook is debugging/validation output only. It is not read by any production pipeline script.

Three-layer cascade, stopping at the first hit per record:

| Layer | Key | Condition |
|---|---|---|
| 1 | `norm_author \|\| norm_title` | `author` and `original_title` both non-empty |
| 2 | `norm_title` | `original_title` non-empty |
| 3 | — | unmatched |

Normalization imported from `src/data/text_utils.py` — identical to `build_goodreads_lookup.py` and `build_matched_dataset.py`.

Tie-breaking among multiple candidates: pick the Goodreads book whose `publication_year` is closest to `czech_pub_year − 3`. Metadata fetched in one pass over `goodreads_books.json.gz`.

In [1]:
import gzip
import json
import sys
from pathlib import Path

import pandas as pd

REPO     = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INTERIM  = REPO / "data" / "interim"
BOOKS_GZ = REPO / "data" / "raw" / "goodreads_books.json.gz"

sys.path.insert(0, str(REPO / "src" / "data"))
from text_utils import normalize, norm_nkc_author

## Load lookups and sample NKC records

In [2]:
print("Loading Goodreads lookup files …", flush=True)
author_names    = json.loads((INTERIM / "goodreads_author_names.json").read_text())
by_author_title = json.loads((INTERIM / "goodreads_by_author_title.json").read_text())
by_title        = json.loads((INTERIM / "goodreads_by_title.json").read_text())
print(f"  author_names    : {len(author_names):>8,} entries")
print(f"  by_author_title : {len(by_author_title):>8,} keys")
print(f"  by_title        : {len(by_title):>8,} keys")

Loading Goodreads lookup files …
  author_names    :  829,524 entries
  by_author_title : 2,679,278 keys
  by_title        : 1,695,226 keys


In [3]:
nkc = pd.read_csv(INTERIM / "nkc_translations.csv", dtype=str).fillna("")
nkc["year"] = pd.to_numeric(nkc["czech_pub_year"], errors="coerce")

pool   = nkc[nkc["source_lang"].str.startswith("eng") & (nkc["year"] >= 2003)]
sample = pool.sample(500, random_state=42).reset_index(drop=True)

print(f"NKC English 2003+ pool : {len(pool):>7,}")
print(f"Sample                 : {len(sample):>7,}")
print(f"  with original_title  : {(sample['original_title'].str.strip() != '').sum():>7,}")
print(f"  with author          : {(sample['author'].str.strip() != '').sum():>7,}")

NKC English 2003+ pool :  62,216
Sample                 :     500
  with original_title  :     439
  with author          :     450


## Matching cascade (layers 1 → 2)

In [4]:
pending = []

for _, row in sample.iterrows():
    author     = row["author"].strip()
    orig_title = row["original_title"].strip()
    czech_year = int(row["year"]) if pd.notna(row["year"]) else None

    candidates  = []
    match_layer = 3

    # Layer 1 — author + title
    if author and orig_title:
        key  = norm_nkc_author(author) + "||" + normalize(orig_title)
        hits = by_author_title.get(key, [])
        if hits:
            candidates, match_layer = hits, 1

    # Layer 2 — title only
    if match_layer == 3 and orig_title:
        hits = by_title.get(normalize(orig_title), [])
        if hits:
            candidates, match_layer = hits, 2

    pending.append({
        "nkc_id":         row["nkc_id"],
        "author":         author,
        "original_title": orig_title,
        "czech_pub_year": czech_year,
        "candidates":     candidates,
        "match_layer":    match_layer,
    })

counts = {1: 0, 2: 0, 3: 0}
for r in pending:
    counts[r["match_layer"]] += 1

print("=== Preliminary match counts (before tie-breaking) ===")
print(f"Layer 1 (author + title) : {counts[1]:>4}")
print(f"Layer 2 (title only)     : {counts[2]:>4}")
print(f"Layer 3 (unmatched)      : {counts[3]:>4}")

=== Preliminary match counts (before tie-breaking) ===
Layer 1 (author + title) :  138
Layer 2 (title only)     :   90
Layer 3 (unmatched)      :  272


## Single pass over goodreads_books.json.gz to fetch candidate metadata

In [5]:
all_candidate_ids = set()
for r in pending:
    all_candidate_ids.update(r["candidates"])

print(f"Unique candidate book_ids to fetch: {len(all_candidate_ids):,}", flush=True)

book_meta = {}

with gzip.open(BOOKS_GZ, "rt", encoding="utf-8") as fh:
    for i, line in enumerate(fh):
        if len(book_meta) == len(all_candidate_ids):
            break
        obj = json.loads(line)
        bid = obj.get("book_id", "")
        if bid not in all_candidate_ids:
            continue
        try:
            pub_year = int(obj.get("publication_year") or 0)
        except (ValueError, TypeError):
            pub_year = 0
        book_meta[bid] = {
            "title":          obj.get("title_without_series") or obj.get("title", ""),
            "pub_year":       pub_year,
            "author_ids":     [a["author_id"] for a in obj.get("authors", [])],
            "ratings_count":  int(obj.get("ratings_count") or 0),
            "average_rating": float(obj.get("average_rating") or 0.0),
        }
        if (i + 1) % 500_000 == 0:
            print(f"  {i+1:,} lines scanned, {len(book_meta):,}/{len(all_candidate_ids):,} found …", flush=True)

print(f"Done. Retrieved {len(book_meta):,} / {len(all_candidate_ids):,} book records.")

Unique candidate book_ids to fetch: 2,052
Done. Retrieved 2,052 / 2,052 book records.


## Tie-breaking and final results

In [6]:
def pick_best(candidates: list, target_year: int) -> str:
    if len(candidates) == 1:
        return candidates[0]
    return min(
        candidates,
        key=lambda bid: abs((book_meta.get(bid, {}).get("pub_year") or 0) - target_year)
        if (book_meta.get(bid, {}).get("pub_year") or 0) > 0 else 9999
    )


rows = []
for r in pending:
    if r["match_layer"] == 3:
        rows.append({
            "nkc_id":            r["nkc_id"],
            "author":            r["author"],
            "original_title":    r["original_title"],
            "czech_pub_year":    r["czech_pub_year"],
            "matched_book_id":   "",
            "match_layer":       3,
            "gr_title":          "",
            "gr_author_ids":     "",
            "gr_pub_year":       "",
            "gr_ratings_count":  "",
            "gr_average_rating": "",
        })
        continue

    target_year = (r["czech_pub_year"] or 2010) - 3
    best_id     = pick_best(r["candidates"], target_year)
    meta        = book_meta.get(best_id, {})

    rows.append({
        "nkc_id":            r["nkc_id"],
        "author":            r["author"],
        "original_title":    r["original_title"],
        "czech_pub_year":    r["czech_pub_year"],
        "matched_book_id":   best_id,
        "match_layer":       r["match_layer"],
        "gr_title":          meta.get("title", ""),
        "gr_author_ids":     "|".join(meta.get("author_ids", [])),
        "gr_pub_year":       meta.get("pub_year", "") or "",
        "gr_ratings_count":  meta.get("ratings_count", ""),
        "gr_average_rating": meta.get("average_rating", ""),
    })

results_df = pd.DataFrame(rows)

## Summary

In [7]:
layer_labels = {
    1: "Layer 1 — author + title",
    2: "Layer 2 — title only",
    3: "Layer 3 — unmatched",
}
n = len(results_df)
print(f"=== Match summary (n={n}) ===")
for layer in [1, 2, 3]:
    cnt = (results_df["match_layer"] == layer).sum()
    print(f"  {layer_labels[layer]:<30} : {cnt:>4}  ({cnt/n:.1%})")

=== Match summary (n=500) ===
  Layer 1 — author + title       :  138  (27.6%)
  Layer 2 — title only           :   90  (18.0%)
  Layer 3 — unmatched            :  272  (54.4%)


## Save results

In [8]:
out_path = INTERIM / "matching_spike_results.csv"
results_df.to_csv(out_path, index=False)
print(f"Saved {len(results_df):,} rows → {out_path}")

Saved 500 rows → /home/firstone/Bachelors-thesis/data/interim/matching_spike_results.csv


## Eyeball: 10 examples per layer

In [9]:
MATCHED_COLS   = ["author", "original_title", "czech_pub_year", "gr_title", "gr_pub_year", "gr_ratings_count"]
UNMATCHED_COLS = ["author", "original_title", "czech_pub_year"]

for layer, label in [(1, "Layer 1 — author + title"), (2, "Layer 2 — title only"), (3, "Layer 3 — unmatched")]:
    sub  = results_df[results_df["match_layer"] == layer]
    rows = sub.sample(min(10, len(sub)), random_state=42)
    cols = UNMATCHED_COLS if layer == 3 else MATCHED_COLS
    print(f"\n{'='*72}")
    print(f" {label}  (total: {len(sub)})")
    print(f"{'='*72}")
    print(rows[cols].to_string(index=False))


 Layer 1 — author + title  (total: 138)
          author                    original_title  czech_pub_year                              gr_title gr_pub_year gr_ratings_count
     Sanders, Ed Poetry and life of Allen Ginsberg            2023 The Poetry and Life of Allen Ginsberg        2000               32
    Dolnick, Ben       At the bottom of everything            2015           At the Bottom of Everything        2013              811
 Steel, Danielle                           Duchess            2019                           The Duchess        2017             1552
    Bellos, Alex    Alex through the looking glass            2016      Alex Through the Looking - Glass                           73
   Francis, Dick                              Bolt            2003                                  Bolt        1988               75
     West, Kasie                      On the fence            2022                          On the Fence        2016               10
  Harris, Thomas     